# Step 5: Final Evaluation and Comparison

This notebook is the culmination of all six steps. Throughout Steps 2–6, we trained and evaluated:

| Step | Model / Experiment | Curriculum |
|---|---|---|
| Step 2 | Logistic Regression (Baseline) | Week 2 |
| Step 3 | Plain MLP | Week 3 |
| Step 4 | MLP + L2 Regularization | Week 4 |
| Step 4 | MLP + Early Stopping | Week 4 |
| Step 6 | MLP + Adam vs SGD+Momentum vs Adam+Cosine | Week 5 |
| Step 6 | He vs Xavier vs Random Initialization | Week 5 |
| Step 6 | Constant LR vs Step Decay vs Cosine Annealing | Week 5 |

### Methodological Goal

We evaluate all models relative to the **class imbalance (37% Malignant vs 63% Benign)** using **Recall** and **F1-Score** as primary metrics — minimizing False Negatives (missed tumors) is the clinical priority.

**Bonus:** ROC/AUC analysis provides threshold-agnostic comparison across all models.

**Execution order dependency:** Run `step4` and `step6` before this notebook so their CSV outputs are available in `outputs/tables/`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_curve, auc

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from data_utils import create_stratified_splits

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_STEP5 = OUTPUTS_DIR / "figures" / "step5"
TABLES_DIR = OUTPUTS_DIR / "tables"

FIGURES_STEP5.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Aggregating All Metrics — Steps 2–6

We load the CSV outputs produced by each previous notebook and consolidate them into a single comparison table.

- `step4_combined_metrics.csv` — Logistic Regression, Plain MLP, MLP+L2, MLP+Early Stopping (Steps 2–4)
- `step6_optimizer_comparison.csv` — Adam vs SGD+Momentum vs Adam+Cosine (Step 6, Part 1)
- `step6_initialization_comparison.csv` — He vs Xavier vs Random inits (Step 6, Part 2)
- `step6_lr_schedule_comparison.csv` — Constant vs Step Decay vs Cosine Annealing (Step 6, Part 3)

In [ ]:
# Load Steps 2-4 metrics
step4_df = pd.read_csv(TABLES_DIR / 'step4_combined_metrics.csv')

step4_df = step4_df.rename(columns={
    'Accuracy'         : 'Accuracy',
    'Precision (Macro)': 'Precision',
    'Recall (Macro)'   : 'Recall',
    'F1-Score (Macro)' : 'F1-Score',
})
step4_df['Group'] = step4_df['Model'].apply(
    lambda x: 'Week 2' if 'Logistic' in x
    else ('Week 3' if 'Plain' in x
    else 'Week 4')
)

# Load Step 6 metrics
def load_step6(filename, group_label, index_col):
    p = TABLES_DIR / filename
    if not p.exists():
        print(f'WARNING: {filename} not found -- run step6 first.')
        return pd.DataFrame()
    df = pd.read_csv(p).rename(columns={index_col: 'Model'})
    for col in ['Accuracy','Precision','Recall','F1-Score']:
        if col in df.columns and df[col].max() > 2:
            df[col] = df[col] / 100
    df['Group'] = group_label
    df['Split'] = 'Validation'
    return df

opt_df   = load_step6('step6_optimizer_comparison.csv',      'Week 5 - Optimizer',     'Label')
init_df  = load_step6('step6_initialization_comparison.csv', 'Week 5 - Initialization', 'Label')
sched_df = load_step6('step6_lr_schedule_comparison.csv',    'Week 5 - LR Schedule',   'Label')

all_metrics = pd.concat(
    [df for df in [step4_df, opt_df, init_df, sched_df] if not df.empty],
    ignore_index=True
)
all_metrics = all_metrics[['Model','Group','Split','Accuracy','Precision','Recall','F1-Score']]

display_df = all_metrics.copy()
for col in ['Accuracy','Precision','Recall','F1-Score']:
    display_df[col] = (display_df[col] * 100).round(2).astype(str) + '%'

print('Full Model Comparison Table (All Steps)')
display(display_df)

all_metrics.to_csv(TABLES_DIR / 'step5_master_comparison.csv', index=False)
print('Saved to outputs/tables/step5_master_comparison.csv')

## 2. ROC Curve and AUC Analysis

ROC/AUC provides threshold-agnostic evaluation — it measures how well each model separates classes across **all possible probability thresholds**, not just the default 0.5 cutoff.

We include:
- Logistic Regression (Week 2 baseline)
- Plain MLP (Week 3 — overfitted)
- MLP + L2 (Week 4 — regularized)
- **MLP + Adam + Cosine + He Init** (Step 6 — Week 5 optimal config, PyTorch)

A model with AUC ≈ 1.0 means it produces a near-perfect ranking of samples by confidence — regardless of threshold."

In [ ]:
group_order  = ['Week 2','Week 3','Week 4','Week 5 - Optimizer','Week 5 - Initialization','Week 5 - LR Schedule']
group_colors = ['#2ecc71','#e74c3c','#3498db','#9b59b6','#f39c12','#1abc9c']
color_map = dict(zip(group_order, group_colors))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, metric in zip(axes, ['F1-Score', 'Recall']):
    for group in group_order:
        subset = all_metrics[all_metrics['Group'] == group]
        if subset.empty:
            continue
        ax.barh(
            subset['Model'],
            subset[metric] * 100,
            color=color_map[group],
            label=group,
            edgecolor='white',
            linewidth=0.4
        )
    ax.set_xlim(95, 100.5)
    ax.set_xlabel(f'{metric} (%)', fontsize=11)
    ax.set_title(f'{metric} -- All Models (Steps 2-6)', fontsize=12)
    ax.legend(title='Curriculum Group', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    ax.grid(axis='x', alpha=0.3)
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(FIGURES_STEP5 / 'final_metric_comparisons.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

ds = create_stratified_splits()
X_tr, y_tr = ds.X_train_scaled, ds.y_train
X_v,  y_v  = ds.X_val_scaled,   ds.y_val

# sklearn models (Steps 2-4)
logreg    = LogisticRegression(random_state=42, max_iter=1000).fit(X_tr, y_tr)
mlp_plain = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                           random_state=42, max_iter=500).fit(X_tr, y_tr)
mlp_l2    = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                           alpha=0.1, random_state=42, max_iter=500).fit(X_tr, y_tr)

# Step 6 optimal: PyTorch MLP -- Adam + Cosine Annealing + He init
X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr.values, dtype=torch.float32).unsqueeze(1)
X_v_t  = torch.tensor(X_v,  dtype=torch.float32)

torch.manual_seed(42)
mlp_step6 = nn.Sequential(
    nn.Linear(30, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),  nn.Sigmoid()
)
for layer in mlp_step6:
    if isinstance(layer, nn.Linear):
        nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
        nn.init.zeros_(layer.bias)

opt6   = optim.Adam(mlp_step6.parameters(), lr=0.001)
sched6 = optim.lr_scheduler.CosineAnnealingLR(opt6, T_max=150, eta_min=1e-5)
crit6  = nn.BCELoss()
train_ds6     = torch.utils.data.TensorDataset(X_tr_t, y_tr_t)
train_loader6 = torch.utils.data.DataLoader(
    train_ds6, batch_size=32, shuffle=True,
    generator=torch.Generator().manual_seed(42)
)

for epoch in range(150):
    mlp_step6.train()
    for xb, yb in train_loader6:
        opt6.zero_grad()
        loss = crit6(mlp_step6(xb), yb)
        l2   = sum(p.pow(2).sum() for p in mlp_step6.parameters())
        (loss + 0.05 * l2).backward()
        opt6.step()
    sched6.step()

mlp_step6.eval()
with torch.no_grad():
    step6_probs = mlp_step6(X_v_t).squeeze().numpy()

print('All models ready -- generating ROC curves.')

# ROC Curves
roc_models = {
    'Logistic Regression (W2)'     : ('sklearn', logreg),
    'Plain MLP -- overfitted (W3)' : ('sklearn', mlp_plain),
    'MLP + L2 (W4)'               : ('sklearn', mlp_l2),
    'MLP + Adam + Cosine + He (W5)': ('pytorch', step6_probs),
}

styles = ['-', '--', '-.', ':']
colors = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6']

plt.figure(figsize=(8, 7))

for (name, (mtype, obj)), style, color in zip(roc_models.items(), styles, colors):
    probs = obj.predict_proba(X_v)[:, 0] if mtype == 'sklearn' else (1 - obj)
    fpr, tpr, _ = roc_curve(y_v, probs, pos_label=0)
    roc_auc = auc(fpr, tpr)
    lw = 2.5 if 'W5' in name else 1.8
    plt.plot(fpr, tpr, lw=lw, linestyle=style, color=color,
             label=f'{name}  (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], color='grey', lw=1, linestyle=':', label='Random Guess')
plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=11)
plt.ylabel('True Positive Rate (Recall)', fontsize=11)
plt.title('ROC Curve Comparison -- Steps 2 through 6\nMalignant Detection Focus', pad=15)
plt.legend(loc='lower right', fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig(FIGURES_STEP5 / 'roc_curve_comparison.png', dpi=200)
plt.show()

## 3. Final Academic Conclusions — Steps 2 through 6

### Week 2 → 3 → 4 Arc

**The Complexity Trap (Week 3):** The Plain MLP proved that adding hidden layers without constraint degrades generalization. The unconstrained optimizer assigned large weights to multicollinear features (correlations >0.99, Step 1), causing Recall to drop — the most dangerous failure mode in a cancer detection context.

**Regularization Restores the Baseline (Week 4):** Both L2 Weight Decay and Early Stopping recovered Recall to the Logistic Regression ceiling. This is not a coincidence — it is the expected mathematical outcome: regularization constrains the effective capacity of the network, preventing it from memorizing training noise.

**The Baseline Reality Check (Week 2):** The ROC/AUC analysis confirms that Logistic Regression achieves AUC ≈ 0.999. The dataset is geometrically simple enough that a linear decision boundary captures nearly all discriminative information. Architectural complexity is not justified by the data's intrinsic structure.

---

### Week 5 Arc — Optimizer, Initialization, LR Schedule

**Optimizer (Step 6, Part 1):** Adam outperforms SGD+Momentum on this dataset because the 30 features have unequal gradient magnitudes (target correlations range from ~0.01 to ~0.82). Adam's per-parameter adaptive learning rate handles this asymmetry; SGD+Momentum cannot. Adam + Cosine Annealing produces the smoothest validation loss curve — the scheduled LR decay prevents overshooting near convergence on a 341-sample training set.

**Initialization (Step 6, Part 2):** He/Kaiming initialization converges faster than Xavier for ReLU networks because it accounts for the factor-of-2 variance loss from ReLU zeroing out half of neurons. Small/Large Random initializations cause vanishing/exploding activation variance, producing slow or unstable convergence.

**LR Schedule (Step 6, Part 3):** Cosine Annealing produces the most stable validation loss. A constant LR causes late-epoch oscillations; step decay helps but introduces discrete instability at each step boundary. The smooth continuous decay of cosine annealing is the best-performing schedule on small datasets.

---

### Final Verdict

The optimal configuration for this dataset — empirically confirmed across all experiments:

> **Logistic Regression** is the most cost-effective model (near-perfect AUC, near-zero training cost).  
> **MLP + L2 + Adam + He + Cosine Annealing** is the methodologically richest model — it matches Logistic Regression's performance but demonstrates that all five weeks of curriculum can be applied coherently and deliberately to a single problem."